# 06 — DQ Monitoring Logs: `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

Este notebook calcula e grava, na tabela Delta **compartilhada** `squad1/dq_monitoring_logs`, o resultado das regras de qualidade de dados (DQ) das duas tabelas Silver desta dupla. Todas as duplas do squad1 gravam nessa tabela, cada uma com as regras das suas próprias tabelas, seguindo o mesmo contrato.

## Contrato da tabela (acordado pela squad)
- Schema: `run_id`, `tabela`, `regra`, `status` (`PASS`/`FAIL`), `severidade` (`Critica`/`Aviso`, sem acento), `qtd_registros_falhos`, `qtd_registros_total`, `timestamp_execucao`, `arquivo_origem`.
- **Uma linha por regra por micro-lote.** `arquivo_origem` é o caminho relativo do micro-lote (`real-time-data/AAAA/MM/DD/HHMMSS/<arquivo>.parquet`); a Gold extrai data e hora dele.
- **Append só do que é novo:** uma combinação `tabela + regra + arquivo_origem` que já está no log não é gravada de novo. Reexecutar o notebook não duplica nada.
- **Exceção desta dupla — regras agregadas:** as 6 regras agregadas por pedido (itens R7 e R10; rastreamento R7 a R10) dependem de dados que podem chegar depois (Silver de pedidos, histórico de eventos). Por isso, a cada execução, as linhas dessas 6 regras **das nossas duas tabelas** são recalculadas e substituídas (`replaceWhere` restrito a `tabela + regra`). Linhas de outras duplas nunca são tocadas.
- Técnicas (R1–R5) = `Critica`; negócio (R6–R10) = `Aviso`.

## Regras/KPIs gerados por este notebook

**`ecommerce_itens_pedido`**
1. `tecnica_1_pk_valida` (Critica) — `id_item_pedido` não nulo e não duplicado
2. `tecnica_2_fk_pedido_valido` (Critica) — `id_pedido` existe na Silver de pedidos
3. `tecnica_3_fk_sku_valido` (Critica) — `sku` existe na Silver de produtos
4. `tecnica_4_quantidade_valida` (Critica) — `quantidade` inteira e ≥ 1
5. `tecnica_5_preco_valido` (Critica) — `preco_unitario` > 0
6. `negocio_6_desconto_nao_maior_preco` (Aviso) — desconto não maior que o preço
7. `negocio_7_soma_itens_igual_valor_total` (Aviso, agregada) — soma de todos os itens do pedido bate com o `valor_total` do pedido
8. `negocio_8_desconto_nao_negativo` (Aviso) — desconto não negativo
9. `negocio_9_desconto_dentro_limite` (Aviso) — desconto até 50% do preço
10. `negocio_10_pedido_tem_item` (Aviso, agregada) — todo pedido tem pelo menos 1 item

**`ecommerce_rastreamento_entregas`**
1. `tecnica_1_pk_valida` (Critica) — `id_rastreamento` não nulo e não duplicado
2. `tecnica_2_status_valido` (Critica) — `status_entrega` dentro do domínio
3. `tecnica_3_fk_pedido_valido` (Critica) — `id_pedido_ecommerce` existe na Silver de pedidos
4. `tecnica_4_data_valida` (Critica) — `dt_evento` preenchida e não futura
5. `tecnica_5_codigo_rastreio_valido` (Critica) — código no formato `TRK` + 6 dígitos
6. `negocio_6_ordem_cronologica` (Aviso) — eventos do pedido em ordem cronológica
7. `negocio_7_entregue_tem_evento` (Aviso, agregada) — pedido "Entregue" tem evento "entregue"
8. `negocio_8_prazo_entrega_30_dias` (Aviso, agregada) — até 30 dias entre "coletado" e "entregue"
9. `negocio_9_cancelado_sem_evento` (Aviso, agregada) — pedido cancelado não tem evento de rastreio
10. `negocio_10_transportadora_unica` (Aviso, agregada) — pedido não cancelado tem uma única transportadora

Nenhuma linha é apagada por este notebook: as falhas são apenas contadas e gravadas no log.

In [0]:
# ============================================================
# CÉLULA 1 — SETUP
# DQ MONITORING LOGS - ITENS_PEDIDO e RASTREAMENTO_ENTREGAS
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net"

caminhos = {
    "silver_itens_pedido": f"{base_squad1}/dupla1/silver/ecommerce_itens_pedido",
    "silver_rastreamento": f"{base_squad1}/dupla1/silver/ecommerce_rastreamento_entregas",
    "silver_pedidos": f"{base_squad1}/silver/ecommerce_pedidos",
    "dq_logs": f"{base_squad1}/dq_monitoring_logs",
}

# Nome do arquivo de cada tabela dentro do micro-lote (usado para montar arquivo_origem)
ARQUIVO_ITENS = "ecommerce_itens_pedido.parquet"
ARQUIVO_RASTREAMENTO = "ecommerce_rastreamento.parquet"

def caminho_relativo(col):
    """Normaliza o caminho do micro-lote para o formato do contrato:
    real-time-data/AAAA/MM/DD/HHMMSS/<arquivo>.parquet
    (na Bronze de itens o caminho vem como URL abfss:// completa)."""
    return F.regexp_extract(col, r"(real-time-data/\d{4}/\d{2}/\d{2}/\d{6}/[^/]+)$", 1)

def pasta_do_lote(col):
    """Pasta do micro-lote: real-time-data/AAAA/MM/DD/HHMMSS"""
    return F.regexp_extract(col, r"(real-time-data/\d{4}/\d{2}/\d{2}/\d{6})", 1)

run_id = str(uuid.uuid4())
agora = datetime.now(timezone.utc)

print("Credenciais OAuth preparadas.")
print("DQ Logs (compartilhado):", caminhos["dq_logs"])
print("run_id desta execução:", run_id)

## Etapa 1 — Setup

Credenciais via Secret Scope, caminhos das duas Silvers desta dupla, da Silver de `pedidos` (usada nas regras agregadas) e da tabela compartilhada `dq_monitoring_logs`, na raiz do container `squad1`.

As funções `caminho_relativo` e `pasta_do_lote` padronizam o `arquivo_origem`: a Bronze de `itens_pedido` guarda a URL `abfss://` completa e a de `rastreamento` guarda o caminho relativo, então as duas são convertidas para `real-time-data/AAAA/MM/DD/HHMMSS/<arquivo>.parquet`.

In [0]:
# ============================================================
# CÉLULA 2 — REGRAS DE QUALIDADE POR LINHA (com severidade)
# ============================================================
# Colunas booleanas já gravadas na Silver: True = passou, False = falhou,
# None = regra não avaliada naquela linha (ex.: referência indisponível).

REGRAS_ITENS_PEDIDO = {
    "tecnica_1_pk_valida":                "Critica",
    "tecnica_2_fk_pedido_valido":         "Critica",
    "tecnica_3_fk_sku_valido":            "Critica",
    "tecnica_4_quantidade_valida":        "Critica",
    "tecnica_5_preco_valido":             "Critica",
    "negocio_6_desconto_nao_maior_preco": "Aviso",
    "negocio_8_desconto_nao_negativo":    "Aviso",
    "negocio_9_desconto_dentro_limite":   "Aviso",
}

REGRAS_RASTREAMENTO = {
    "tecnica_1_pk_valida":              "Critica",
    "tecnica_2_status_valido":          "Critica",
    "tecnica_3_fk_pedido_valido":       "Critica",
    "tecnica_4_data_valida":            "Critica",
    "tecnica_5_codigo_rastreio_valido": "Critica",
    "negocio_6_ordem_cronologica":      "Aviso",
}

print("Regras itens_pedido:", list(REGRAS_ITENS_PEDIDO.keys()))
print("Regras rastreamento:", list(REGRAS_RASTREAMENTO.keys()))

## Etapa 2 — Regras de qualidade por linha

Mapeia cada regra booleana já calculada na Silver para a sua severidade, sem acento (`Critica`/`Aviso`), como manda o contrato. Só entram aqui as regras **por linha**; as regras agregadas por pedido ficam na Etapa 4.

In [0]:
# ============================================================
# CÉLULA 3 — FALHAS POR REGRA E POR MICRO-LOTE (regras por linha)
# ============================================================

df_silver_itens = (spark.read.format("delta").options(**adls_options).load(caminhos["silver_itens_pedido"])
    .withColumn("arquivo_origem", caminho_relativo(F.col("bronze_source_file"))))
df_silver_rastreamento = (spark.read.format("delta").options(**adls_options).load(caminhos["silver_rastreamento"])
    .withColumn("arquivo_origem", caminho_relativo(F.col("bronze_source_file"))))

def falhas_por_lote(df, nome_tabela, regras_dict):
    """Uma linha por (regra, micro-lote). Linhas com a regra nula não entram no total."""
    partes = []
    for regra, severidade in regras_dict.items():
        partes.append(df
            .filter(F.col(regra).isNotNull())
            .groupBy("arquivo_origem")
            .agg(F.count("*").alias("qtd_registros_total"),
                 F.sum(F.when(F.col(regra) == False, 1).otherwise(0)).cast("long").alias("qtd_registros_falhos"))
            .withColumn("tabela", F.lit(nome_tabela))
            .withColumn("regra", F.lit(regra))
            .withColumn("severidade", F.lit(severidade)))
    resultado = partes[0]
    for p in partes[1:]:
        resultado = resultado.unionByName(p)
    return resultado

df_linhas = (falhas_por_lote(df_silver_itens, "ecommerce_itens_pedido", REGRAS_ITENS_PEDIDO)
    .unionByName(falhas_por_lote(df_silver_rastreamento, "ecommerce_rastreamento_entregas", REGRAS_RASTREAMENTO)))

# Garante que todo micro-lote foi reconhecido no formato do contrato
sem_lote = df_linhas.filter(F.col("arquivo_origem") == "").count()
assert sem_lote == 0, f"{sem_lote} linhas com bronze_source_file fora do padrão real-time-data/AAAA/MM/DD/HHMMSS/"

print("Linhas (regra × micro-lote) das regras por linha:", df_linhas.count())

## Etapa 3 — Falhas por regra e por micro-lote (regras por linha)

Para cada regra, agrupa a Silver por `arquivo_origem` (o micro-lote de onde a linha veio) e conta o total avaliado e as falhas (`False`). Linhas em que a regra está nula (não avaliada) ficam fora do total. O resultado tem **uma linha por regra por micro-lote**, como pede o contrato. O `assert` garante que nenhum caminho fora do padrão escape para o log.

In [0]:
# ============================================================
# CÉLULA 4 — REGRAS AGREGADAS (por pedido), atribuídas a cada micro-lote
# ============================================================
# A regra é avaliada por pedido com todo o histórico disponível e o resultado
# é atribuído ao micro-lote em que o pedido (ou o evento) apareceu.

# Estado mais recente de cada pedido na Silver da Dupla 4 (um pedido pode chegar em vários lotes)
janela_ultimo = Window.partitionBy("id_pedido").orderBy(F.col("bronze_source_file").desc())
df_pedidos_lote = (spark.read.format("delta").options(**adls_options).load(caminhos["silver_pedidos"])
    .withColumn("pasta_lote", pasta_do_lote(F.col("bronze_source_file"))))
df_pedidos = (df_pedidos_lote
    .withColumn("_ordem", F.row_number().over(janela_ultimo))
    .filter(F.col("_ordem") == 1)
    .select("id_pedido", "status_pedido", "valor_total"))

def linha_agregada(df_por_lote, nome_tabela, regra):
    """df_por_lote: arquivo_origem, qtd_registros_total, qtd_registros_falhos"""
    return (df_por_lote
        .withColumn("tabela", F.lit(nome_tabela))
        .withColumn("regra", F.lit(regra))
        .withColumn("severidade", F.lit("Aviso"))
        .select("arquivo_origem", "qtd_registros_total",
                F.col("qtd_registros_falhos").cast("long").alias("qtd_registros_falhos"),
                "tabela", "regra", "severidade"))

agregadas = []

# --- ITENS: Regra 7 — soma de TODOS os itens do pedido bate com valor_total ---
soma_pedido = (df_silver_itens
    .withColumn("valor_item", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"))
    .groupBy("id_pedido").agg(F.sum("valor_item").alias("soma_calculada"))
    .join(df_pedidos.select("id_pedido", "valor_total"), "id_pedido", "left")
    .withColumn("diverge", F.col("valor_total").isNull() | (F.abs(F.col("soma_calculada") - F.col("valor_total")) > 0.01)))
r7_itens = (df_silver_itens.select("arquivo_origem", "id_pedido").distinct()
    .join(soma_pedido.select("id_pedido", "diverge"), "id_pedido", "left")
    .groupBy("arquivo_origem")
    .agg(F.count("*").alias("qtd_registros_total"),
         F.sum(F.when(F.col("diverge"), 1).otherwise(0)).alias("qtd_registros_falhos")))
agregadas.append(linha_agregada(r7_itens, "ecommerce_itens_pedido", "negocio_7_soma_itens_igual_valor_total"))

# --- ITENS: Regra 10 — todo pedido tem pelo menos 1 item ---
# Avaliada nos pedidos de cada micro-lote; o log usa o arquivo de itens da mesma pasta.
pedidos_com_item = df_silver_itens.select("id_pedido").distinct().withColumn("tem_item", F.lit(True))
r10_itens = (df_pedidos_lote.select("pasta_lote", "id_pedido").distinct()
    .join(pedidos_com_item, "id_pedido", "left")
    .withColumn("arquivo_origem", F.concat_ws("/", "pasta_lote", F.lit(ARQUIVO_ITENS)))
    .groupBy("arquivo_origem")
    .agg(F.count("*").alias("qtd_registros_total"),
         F.sum(F.when(F.col("tem_item").isNull(), 1).otherwise(0)).alias("qtd_registros_falhos")))
agregadas.append(linha_agregada(r10_itens, "ecommerce_itens_pedido", "negocio_10_pedido_tem_item"))

# --- RASTREAMENTO: Regra 7 — pedido 'Entregue' tem evento 'entregue' ---
pedidos_evento_entregue = (df_silver_rastreamento.filter(F.col("status_entrega") == "entregue")
    .select(F.col("id_pedido_ecommerce").alias("id_pedido")).distinct().withColumn("tem_evento", F.lit(True)))
r7_rastr = (df_pedidos_lote.select("pasta_lote", "id_pedido").distinct()
    .join(df_pedidos.filter(F.col("status_pedido") == "Entregue").select("id_pedido"), "id_pedido", "inner")
    .join(pedidos_evento_entregue, "id_pedido", "left")
    .withColumn("arquivo_origem", F.concat_ws("/", "pasta_lote", F.lit(ARQUIVO_RASTREAMENTO)))
    .groupBy("arquivo_origem")
    .agg(F.count("*").alias("qtd_registros_total"),
         F.sum(F.when(F.col("tem_evento").isNull(), 1).otherwise(0)).alias("qtd_registros_falhos")))
agregadas.append(linha_agregada(r7_rastr, "ecommerce_rastreamento_entregas", "negocio_7_entregue_tem_evento"))

# --- RASTREAMENTO: Regra 8 — até 30 dias entre 'coletado' e 'entregue' ---
coletado = (df_silver_rastreamento.filter(F.col("status_entrega") == "coletado")
    .groupBy("id_pedido_ecommerce").agg(F.min("dt_evento").alias("dt_coletado")))
r8_rastr = (df_silver_rastreamento.filter(F.col("status_entrega") == "entregue")
    .select("arquivo_origem", "id_pedido_ecommerce", F.col("dt_evento").alias("dt_entregue"))
    .join(coletado, "id_pedido_ecommerce", "inner")
    .withColumn("fora_prazo", F.datediff("dt_entregue", "dt_coletado") > 30)
    .groupBy("arquivo_origem")
    .agg(F.count("*").alias("qtd_registros_total"),
         F.sum(F.when(F.col("fora_prazo"), 1).otherwise(0)).alias("qtd_registros_falhos")))
agregadas.append(linha_agregada(r8_rastr, "ecommerce_rastreamento_entregas", "negocio_8_prazo_entrega_30_dias"))

# --- RASTREAMENTO: Regra 9 — pedido cancelado não tem evento de rastreio ---
cancelados = df_pedidos.filter(F.col("status_pedido") == "Cancelado").select("id_pedido").withColumn("cancelado", F.lit(True))
r9_rastr = (df_silver_rastreamento.select("arquivo_origem", F.col("id_pedido_ecommerce").alias("id_pedido")).distinct()
    .join(cancelados, "id_pedido", "left")
    .groupBy("arquivo_origem")
    .agg(F.count("*").alias("qtd_registros_total"),
         F.sum(F.when(F.col("cancelado"), 1).otherwise(0)).alias("qtd_registros_falhos")))
agregadas.append(linha_agregada(r9_rastr, "ecommerce_rastreamento_entregas", "negocio_9_cancelado_sem_evento"))

# --- RASTREAMENTO: Regra 10 — pedido não cancelado tem uma única transportadora ---
qtd_transp = (df_silver_rastreamento
    .groupBy(F.col("id_pedido_ecommerce").alias("id_pedido"))
    .agg(F.countDistinct("id_transportadora").alias("qtd_transportadoras")))
r10_rastr = (df_silver_rastreamento.select("arquivo_origem", F.col("id_pedido_ecommerce").alias("id_pedido")).distinct()
    .join(cancelados, "id_pedido", "left_anti")
    .join(qtd_transp, "id_pedido", "left")
    .groupBy("arquivo_origem")
    .agg(F.count("*").alias("qtd_registros_total"),
         F.sum(F.when(F.col("qtd_transportadoras") > 1, 1).otherwise(0)).alias("qtd_registros_falhos")))
agregadas.append(linha_agregada(r10_rastr, "ecommerce_rastreamento_entregas", "negocio_10_transportadora_unica"))

df_agregadas = agregadas[0]
for a in agregadas[1:]:
    df_agregadas = df_agregadas.unionByName(a)

print("Linhas (regra × micro-lote) das regras agregadas:", df_agregadas.count())

## Etapa 4 — Regras agregadas (por pedido)

Regras que dependem de agregação por pedido. Cada uma é avaliada **com todo o histórico disponível** (por exemplo, a soma de todos os itens de um pedido) e o resultado é atribuído ao micro-lote em que o pedido ou o evento apareceu, para que também tenham `arquivo_origem`:

- **Itens R7** e **rastreamento R9/R10**: pedidos que aparecem no micro-lote da própria tabela.
- **Itens R10** e **rastreamento R7**: partem dos pedidos de cada micro-lote da Silver de `pedidos` e usam o arquivo da tabela desta dupla na mesma pasta (`.../HHMMSS/ecommerce_itens_pedido.parquet` ou `ecommerce_rastreamento.parquet`).
- **Rastreamento R8**: micro-lote em que chegou o evento "entregue".

O status de cada pedido vem da versão mais recente dele na Silver de `pedidos`. Todas as junções usam `join`, sem `collect()` para o driver.

Como o resultado dessas regras pode mudar quando outros dados chegam, elas **não** seguem o "append só do que é novo": a Etapa 6 substitui as linhas delas a cada execução, para que o log sempre reflita o estado atual.

In [0]:
# ============================================================
# CÉLULA 5 — MONTAR O DATAFRAME DE LOGS NO SCHEMA DO CONTRATO
# ============================================================

COLUNAS_LOG = ["run_id", "tabela", "regra", "status", "severidade",
               "qtd_registros_falhos", "qtd_registros_total", "timestamp_execucao", "arquivo_origem"]

df_novo = (df_linhas.unionByName(df_agregadas)
    .filter(F.col("qtd_registros_total") > 0)
    .withColumn("run_id", F.lit(run_id))
    .withColumn("status", F.when(F.col("qtd_registros_falhos") > 0, "FAIL").otherwise("PASS"))
    .withColumn("qtd_registros_total", F.col("qtd_registros_total").cast("long"))
    .withColumn("timestamp_execucao", F.lit(agora).cast("timestamp"))
    .select(*COLUNAS_LOG))

print("===== LOGS CALCULADOS NESTA EXECUÇÃO =====")
display(df_novo.orderBy("tabela", "regra", "arquivo_origem"))

## Etapa 5 — Schema do contrato

Junta as regras por linha e as agregadas e monta exatamente as 9 colunas do contrato, na mesma ordem e com os mesmos tipos da tabela compartilhada (`qtd_*` como `long`, `timestamp_execucao` como `timestamp`). Combinações sem nenhum registro avaliado (`qtd_registros_total = 0`) não são gravadas.

In [0]:
# ============================================================
# CÉLULA 6 — GRAVAR NO DQ_MONITORING_LOGS
#   (a) regras por linha: append só do que é novo
#   (b) regras agregadas: recalculadas e substituídas a cada execução
# ============================================================

REGRAS_AGREGADAS = {
    "ecommerce_itens_pedido": ["negocio_7_soma_itens_igual_valor_total", "negocio_10_pedido_tem_item"],
    "ecommerce_rastreamento_entregas": ["negocio_7_entregue_tem_evento", "negocio_8_prazo_entrega_30_dias",
                                        "negocio_9_cancelado_sem_evento", "negocio_10_transportadora_unica"],
}
pares_agregados = [(t, r) for t, regras in REGRAS_AGREGADAS.items() for r in regras]
cond_agregadas = "(" + " OR ".join(f"(tabela = '{t}' AND regra = '{r}')" for t, r in pares_agregados) + ")"
eh_agregada = F.expr(cond_agregadas)  # mesmo critério usado no replaceWhere

df_novo_linha = df_novo.filter(~eh_agregada)
df_novo_agregada = df_novo.filter(eh_agregada).select(*COLUNAS_LOG)

# (a) Regras por linha — append só das combinações tabela + regra + arquivo que ainda não existem
chaves_existentes = (spark.read.format("delta").options(**adls_options).load(caminhos["dq_logs"])
    .filter(F.col("tabela").isin("ecommerce_itens_pedido", "ecommerce_rastreamento_entregas"))
    .filter(F.col("arquivo_origem").isNotNull())
    .select("tabela", "regra", "arquivo_origem")
    .distinct())

df_gravar = df_novo_linha.join(chaves_existentes, ["tabela", "regra", "arquivo_origem"], "left_anti").select(*COLUNAS_LOG)
qtd_gravar = df_gravar.count()

if qtd_gravar > 0:
    (df_gravar.write
        .format("delta")
        .mode("append")
        .options(**adls_options)
        .save(caminhos["dq_logs"]))
    print(f"[OK] Regras por linha: {qtd_gravar} linhas novas gravadas (run_id={run_id}).")
else:
    print("[OK] Regras por linha: nenhuma combinação nova de tabela + regra + arquivo. Nada foi gravado.")

# (b) Regras agregadas — substitui só as linhas dessas 6 regras das nossas duas tabelas
qtd_agregadas = df_novo_agregada.count()
qtd_antes = (spark.read.format("delta").options(**adls_options).load(caminhos["dq_logs"])
    .filter(cond_agregadas).count())

if qtd_agregadas > 0:
    (df_novo_agregada.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", cond_agregadas)
        .options(**adls_options)
        .save(caminhos["dq_logs"]))
    print(f"[OK] Regras agregadas: {qtd_antes} linhas substituídas por {qtd_agregadas} recalculadas (run_id={run_id}).")
else:
    print("[AVISO] Regras agregadas: nenhum resultado calculado nesta execução. As linhas existentes foram mantidas.")

## Etapa 6 — Gravação no `dq_monitoring_logs` (compartilhado)

A gravação tem duas partes:

**(a) Regras por linha (14 regras):** antes de gravar, descarta (`left_anti`) toda combinação `tabela + regra + arquivo_origem` que já existe no log para as tabelas desta dupla. Reexecutar não duplica: só entram micro-lotes novos. Escrita em `append`.

**(b) Regras agregadas (6 regras):** o resultado delas pode mudar quando chegam dados de outras tabelas (por exemplo, a regra 7 de rastreamento ficou congelada em 100% de falha enquanto a Silver de rastreamento estava incompleta). Por isso, a cada execução, as linhas dessas regras são **substituídas** pelo resultado recalculado, com `overwrite` + `replaceWhere` restrito aos pares `tabela + regra` desta dupla. Nenhuma linha de outra dupla, nem das regras por linha, é tocada. Se o cálculo não gerar nenhuma linha, nada é substituído.

Nas duas partes a escrita é feita **sem `mergeSchema`**: se o DataFrame não bater com o schema da tabela compartilhada, a gravação falha em vez de alterar a estrutura que as outras duplas usam.

In [0]:
# ============================================================
# CÉLULA 7 — VALIDAÇÃO FINAL
# ============================================================

df_logs_final = spark.read.format("delta").options(**adls_options).load(caminhos["dq_logs"])
df_nossas_tabelas = df_logs_final.filter(F.col("tabela").isin("ecommerce_itens_pedido", "ecommerce_rastreamento_entregas"))

print("===== LINHAS GRAVADAS NESTA EXECUÇÃO =====")
display(df_nossas_tabelas.filter(F.col("run_id") == run_id).orderBy("tabela", "regra", "arquivo_origem"))

# 1. Nenhuma combinação tabela + regra + arquivo repetida (considerando só linhas no contrato)
duplicadas = (df_nossas_tabelas.filter(F.col("arquivo_origem").isNotNull())
    .groupBy("tabela", "regra", "arquivo_origem").count()
    .filter(F.col("count") > 1).count())

# 2. Severidade sempre sem acento nas linhas novas
sev_fora = (df_nossas_tabelas.filter(F.col("run_id") == run_id)
    .filter(~F.col("severidade").isin("Critica", "Aviso")).count())

# 3. Regras agregadas: todas as linhas devem ser desta execução (foram substituídas agora)
agregadas_antigas = (df_logs_final.filter(cond_agregadas)
    .filter(F.col("run_id") != run_id).count()) if qtd_agregadas > 0 else 0

# 4. Linhas antigas fora do contrato (gravadas antes desta correção, sem arquivo_origem)
legado = df_nossas_tabelas.filter(F.col("arquivo_origem").isNull()).count()

print(f"Combinações duplicadas: {duplicadas}")
print(f"Severidade fora do padrão nesta execução: {sev_fora}")
print(f"Regras agregadas com linhas de execuções anteriores: {agregadas_antigas}")
print(f"Linhas antigas sem arquivo_origem (a limpar em acordo com a squad): {legado}")

assert duplicadas == 0, "Há combinações tabela + regra + arquivo_origem duplicadas no log."
assert sev_fora == 0, "Há severidade fora de Critica/Aviso."
assert agregadas_antigas == 0, "Há linhas de regras agregadas que não foram substituídas nesta execução."

## Etapa 7 — Validação final

Relê a tabela compartilhada e confere, para as tabelas desta dupla:
1. que nenhuma combinação `tabela + regra + arquivo_origem` está repetida;
2. que a severidade gravada nesta execução é `Critica` ou `Aviso`, sem acento;
3. que todas as linhas das regras agregadas são desta execução (ou seja, foram de fato substituídas);
4. quantas linhas antigas ainda estão sem `arquivo_origem`. Elas foram gravadas pela versão anterior deste notebook e devem ser removidas **em acordo com a squad**, porque a tabela é compartilhada. As Golds que leem o log devem ignorar linhas com `arquivo_origem` nulo.